# 04 · Validación

Chequeos de consistencia sobre el modelo de datos (`data/processed/vivienda.db`). Deja
constancia de que los números del análisis son sólidos: cuadran con la fuente, tienen cobertura completa (ningún crédito o anuncio se queda sin segmento) y respetan las decisiones documentadas. Cada bloque termina con un `assert`; si algo
falla, el notebook se detendrá.

**Nota sobre la numeración.** Los chequeos se etiquetan con letras (A–F), no con números, para distinguirlos de los bloques de análisis del `03_eda`, que van numerados (1–4): las letras indican que son verificaciones de consistencia, no etapas del estudio.

Se requiere haber corrido `02_modelo_datos.ipynb`.

## Configuración

In [1]:
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd

REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent

con = sqlite3.connect(REPO / "data" / "processed" / "vivienda.db")
sniiv = pd.read_sql("SELECT * FROM sniiv", con)
segmento_valor = pd.read_sql("SELECT * FROM segmento_valor", con)
inmuebles = pd.read_sql("SELECT * FROM inmuebles", con)
con.close()

ORDEN = ["Económica", "Popular", "Tradicional", "Media", "Residencial", "Residencial plus"]

## A. Totales contra la fuente

Los totales del modelo deben coincidir con los del cubo SNIIV: **69,448** créditos y
**\$24,493,507,058.80** de monto.

In [2]:
tot_acc = int(sniiv["acciones"].sum())
tot_mon = round(float(sniiv["monto"].sum()), 2)
assert tot_acc == 69_448, tot_acc
assert tot_mon == 24_493_507_058.80, tot_mon
print(f"A. OK — {tot_acc:,} acciones; ${tot_mon:,.2f}")

A. OK — 69,448 acciones; $24,493,507,058.80


## B. Las participaciones suman 100% por año

In [3]:
clasif = sniiv[sniiv["segmento"] != "No disponible"]
sh = clasif.pivot_table(index="anio", columns="segmento", values="acciones", aggfunc="sum")
sh = sh.div(sh.sum(axis=1), axis=0) * 100
assert np.allclose(sh.sum(axis=1), 100)
print(f"B. OK — participaciones suman 100% en los {sh.shape[0]} años")

B. OK — participaciones suman 100% en los 11 años


## C. Cobertura del mapeo

Todo segmento clasificado debe tener rango de valor; "No disponible" no debe tenerlo.

In [4]:
j = sniiv.merge(segmento_valor, on=["anio", "segmento"], how="left")
sin_rango = j[(j["segmento"] != "No disponible") & (j["valor_inf"].isna())]
nd_con_rango = j[(j["segmento"] == "No disponible") & (j["valor_inf"].notna())]
assert len(sin_rango) == 0 and len(nd_con_rango) == 0
print("C. OK — todo clasificado tiene rango; 'No disponible' no")

C. OK — todo clasificado tiene rango; 'No disponible' no


## D. Integridad del mapeo

66 filas (11 años × 6 segmentos); económica arranca en 0; residencial plus es abierto; y las
fronteras son contiguas (sin huecos ni traslapes).

In [5]:
assert len(segmento_valor) == 66
for anio, g in segmento_valor.groupby("anio"):
    g = g.set_index("segmento").reindex(ORDEN)
    assert g.loc["Económica", "valor_inf"] == 0
    assert pd.isna(g.loc["Residencial plus", "valor_sup"])
    assert np.allclose(g["valor_sup"].values[:-1], g["valor_inf"].values[1:])
print("D. OK — 66 filas, extremos correctos y fronteras contiguas")

D. OK — 66 filas, extremos correctos y fronteras contiguas


## E. Asignación de segmento a los listados

In [6]:
b25 = segmento_valor[segmento_valor["anio"] == 2025].set_index("segmento").reindex(ORDEN)
bordes = [0] + list(b25["valor_inf"].iloc[1:].values) + [np.inf]
seg = pd.cut(inmuebles["precio"], bins=bordes, labels=ORDEN, right=False)
assert seg.notna().all()
print(f"E. OK — los {len(inmuebles)} anuncios quedaron asignados a un segmento, sin huecos")

E. OK — los 60 anuncios quedaron asignados a un segmento, sin huecos


## F. Sanidad de valores

In [7]:
assert (sniiv["monto"] >= 0).all() and (sniiv["acciones"] >= 0).all()
assert (inmuebles["precio"] > 0).all()
print("F. OK — sin montos/acciones negativos ni precios no positivos")

F. OK — sin montos/acciones negativos ni precios no positivos


## Resultado

In [8]:
print("Validación: OK — el modelo de datos es internamente consistente y cuadra con la fuente.")

Validación: OK — el modelo de datos es internamente consistente y cuadra con la fuente.
